# NCA Parent ML — Annual z-score IA45 + depth-weighted 0–30 cm POLARIS

This notebook repeats the controlled parent-classification workflow using a more parsimonious
static edaphic representation.

The three representations are:

1. **Annual Z only** — 45 annual z-scored IA45 optical features.
2. **POLARIS 0–30 cm only** — seven depth-integrated soil predictors.
3. **Annual Z + POLARIS 0–30 cm** — the 45 optical features plus the seven static soil predictors.

The POLARIS predictors are integrated across the `0–5`, `5–15`, and `15–30 cm`
layers using thickness weights of `5`, `10`, and `15 cm`, respectively.

For `sand`, `silt`, `clay`, `bd`, `theta_s`, and `ph`, the notebook uses a
thickness-weighted arithmetic mean. `om_log10` is first returned to linear organic-matter
space, averaged by depth thickness, and transformed back to `log10` so that the aggregate
represents the log of the arithmetic mean rather than the mean of logged values.

This notebook preserves the same vegetation rows, grouped folds, learners, annual optical
z-score parameters, mapping model, uncertainty outputs, and evaluation logic as the prior
POLARIS texture-depth experiment. The only intended experimental change is the POLARIS
feature representation.


In [10]:
# =============================================================================
# 0. CONFIGURATION
#
# REQUIRED USER EDIT:
#     Change POLARIS_SOURCE to the local POLARIS VRT or multiband TIFF.
#
# If the VRT itself does not carry useful band descriptions, Section 2 will
# inspect source shards referenced by the VRT and recover band names there.
# =============================================================================

from pathlib import Path
import re
import time
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 280)

POLARIS_SOURCE = Path(
    r"C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif"
)

OPTICAL_SAMPLE_DIR = Path(
    r"C:\NCA_DATA\Analysis\Static_Optical_Sample_IA45"
)

ZSCORE_OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_IA45"
)

ZSCORE_PARAMETER_CSV = (
    ZSCORE_OUTPUT_DIR
    / "annual_IA45_zscore_parameters_S500.csv"
)

VEG_TRAINING_CSV = Path(
    r"A:\NCA_DATA\Validation\Supervised_State_Classification"
    r"\FieldVeg_K2_IA45_Training.csv"
)

HARMONIC_ROOT = Path(
    r"A:\NCA_DATA\S2_Harmonics"
)

KEEP_MASK = Path(
    r"C:\NCA_DATA\templates\Masks\NCA_keepMask_v2.tif"
)

TRUE_NCA_SHP = Path(
    r"C:\NCA_DATA\templates\SRBOP_NCA_extent_26911.shp"
)

OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_POLARIS_0_30Weighted"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

YEARS_TO_MAP = list(range(2016, 2026))
VEG_YEARS = [2016, 2018, 2025]

REFERENCE_TIER = 500
EXPECTED_IA_D = 45

VEG_PARENT_CODES = list(range(1, 9))
VEG_CLASSES = [f"VEG::{i}" for i in VEG_PARENT_CODES]

CLASS_TO_CODE = {
    f"VEG::{i}": i
    for i in VEG_PARENT_CODES
}

CODE_TO_CLASS = {
    code: cls
    for cls, code in CLASS_TO_CODE.items()
}

RANDOM_STATE = 42
N_SPLITS = 5
COMPRESS = "DEFLATE"

MODEL_FOR_MAPPING = "random_forest"
FILTER_TRAINING_TO_MASK = False
POLARIS_RESAMPLING = Resampling.bilinear

RUN_MAPPING = True
SAVE_COMPARISON_FIGURES = True

print("POLARIS source:", POLARIS_SOURCE)
print("Output:", OUTPUT_DIR)


POLARIS source: C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif
Output: C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_POLARIS_0_30Weighted


## POLARIS predictor selection

The source stack contains the seven soil-variable families used here across six native
POLARIS depth intervals. This experiment requests only the upper three layers:

- `0–5 cm`
- `5–15 cm`
- `15–30 cm`

and collapses them to seven integrated `0–30 cm` predictors:

- sand
- silt
- clay
- bulk density (`bd`)
- saturated soil-water content (`theta_s`)
- pH
- organic matter (`om_log10`)

The raw source bands are resolved from their metadata rather than from a hard-coded physical
band order. All 21 source bands are read, then reduced to seven features in a fixed semantic
order.


In [11]:
# =============================================================================
# 0B. POLARIS 0–30 CM VARIABLE SELECTION
# =============================================================================

POLARIS_VARIABLES = [
    "sand",
    "silt",
    "clay",
    "bd",
    "theta_s",
    "ph",
    "om_log10",
]

POLARIS_DEPTHS = [
    "0_5",
    "5_15",
    "15_30",
]

POLARIS_DEPTH_THICKNESS_CM = {
    "0_5": 5.0,
    "5_15": 10.0,
    "15_30": 15.0,
}

POLARIS_VARIABLE_INFO = {
    "sand": "percent sand",
    "silt": "percent silt",
    "clay": "percent clay",
    "bd": "bulk density",
    "theta_s": "saturated soil-water content",
    "ph": "soil pH",
    "om_log10": "organic matter (log10)",
}

POLARIS_AGG_FEATURE_NAMES = [
    f"{variable}_0_30_dw"
    for variable in POLARIS_VARIABLES
]

print("Requested POLARIS 0–30 cm variables:")
for variable in POLARIS_VARIABLES:
    print(
        f"  {variable:10s} — "
        f"{POLARIS_VARIABLE_INFO[variable]}"
    )

print("\nSource depths:", POLARIS_DEPTHS)
print("Thicknesses (cm):", POLARIS_DEPTH_THICKNESS_CM)
print("Aggregated features:", POLARIS_AGG_FEATURE_NAMES)


Requested POLARIS 0–30 cm variables:
  sand       — percent sand
  silt       — percent silt
  clay       — percent clay
  bd         — bulk density
  theta_s    — saturated soil-water content
  ph         — soil pH
  om_log10   — organic matter (log10)

Source depths: ['0_5', '5_15', '15_30']
Thicknesses (cm): {'0_5': 5.0, '5_15': 10.0, '15_30': 15.0}
Aggregated features: ['sand_0_30_dw', 'silt_0_30_dw', 'clay_0_30_dw', 'bd_0_30_dw', 'theta_s_0_30_dw', 'ph_0_30_dw', 'om_log10_0_30_dw']


## 1. Resolve IA45 feature names and annual z-score parameters

The notebook first reuses the S500 annual normalization parameters already
written by the z-score workflow. If that file is missing, it reconstructs the
same weighted S500 parameters from the static optical sample.


In [12]:
# =============================================================================
# 1. IA45 NAMES
# =============================================================================

ANNUAL_METADATA_COLS = {
    "point_id",
    "Year",
    "tile_id",
    "sample_rank",
    "mask_row",
    "mask_col",
    "Easting",
    "Northing",
    "in_s100",
    "in_s250",
    "in_s500",
    "valid_ia45",
}


def annual_sample_path(
    year,
):
    return (
        OPTICAL_SAMPLE_DIR
        / f"static_optical_IA45_{year}.csv.gz"
    )


def annual_k2_path(
    year,
):
    return (
        HARMONIC_ROOT
        / str(
            year
        )
        / (
            f"{year}"
            "_K2_InterceptAmpPhase_nobs_RMSE.tif"
        )
    )


first_annual = annual_sample_path(
    YEARS_TO_MAP[0]
)

if not first_annual.exists():
    raise FileNotFoundError(
        f"Missing annual optical sample:\n{first_annual}"
    )

header = pd.read_csv(
    first_annual,
    compression="gzip",
    nrows=0,
)

IA_COLS = [
    c
    for c in header.columns
    if c not in ANNUAL_METADATA_COLS
]

if len(
    IA_COLS
) != EXPECTED_IA_D:
    raise ValueError(
        f"Expected {EXPECTED_IA_D} IA45 features; "
        f"found {len(IA_COLS)}."
    )

print(
    "IA45 feature count:",
    len(
        IA_COLS
    )
)


IA45 feature count: 45


In [13]:
# =============================================================================
# 1B. LOAD OR RECONSTRUCT S500 Z-SCORE PARAMETERS
# =============================================================================

def weighted_mean_sd(
    x,
    w,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    w = np.asarray(
        w,
        dtype=np.float64,
    )

    valid = (
        np.isfinite(
            x
        )
        & np.isfinite(
            w
        )
        & (
            w > 0
        )
    )

    x = x[
        valid
    ]

    w = w[
        valid
    ]

    weight_sum = np.sum(
        w
    )

    mean = np.sum(
        w
        * x
    ) / weight_sum

    variance = np.sum(
        w
        * (
            x
            - mean
        ) ** 2
    ) / weight_sum

    return (
        float(
            mean
        ),
        float(
            np.sqrt(
                variance
            )
        ),
    )


if ZSCORE_PARAMETER_CSV.exists():

    reference_parameters = pd.read_csv(
        ZSCORE_PARAMETER_CSV
    )

    print(
        "Loaded existing S500 z-score parameters:"
    )

    print(
        ZSCORE_PARAMETER_CSV
    )

else:

    print(
        "Existing S500 parameter file not found."
    )

    print(
        "Reconstructing S500 parameters from the static optical sample."
    )

    TILE_MANIFEST = (
        OPTICAL_SAMPLE_DIR
        / "static_optical_tile_manifest.csv"
    )

    TEMPORAL_COVERAGE = (
        OPTICAL_SAMPLE_DIR
        / "static_optical_sample_temporal_coverage.csv"
    )

    tiles = pd.read_csv(
        TILE_MANIFEST
    )

    sample_support = pd.read_csv(
        TEMPORAL_COVERAGE
    )

    sample_support[
        "persistent_valid"
    ] = (
        sample_support[
            "persistent_valid"
        ]
        .astype(str)
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )

    reference_points = (
        sample_support.loc[
            sample_support[
                "persistent_valid"
            ]
            & sample_support[
                "sample_rank"
            ].le(
                REFERENCE_TIER
            ),
            [
                "point_id",
                "tile_id",
            ],
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    tile_valid_lookup = (
        tiles[
            [
                "tile_id",
                "n_valid_pixels",
            ]
        ]
        .drop_duplicates()
        .set_index(
            "tile_id"
        )[
            "n_valid_pixels"
        ]
        .to_dict()
    )

    per_tile_n = (
        reference_points[
            "tile_id"
        ]
        .value_counts()
        .to_dict()
    )

    reference_points[
        "tile_valid_pixels"
    ] = reference_points[
        "tile_id"
    ].map(
        tile_valid_lookup
    )

    reference_points[
        "n_sample_in_tile"
    ] = reference_points[
        "tile_id"
    ].map(
        per_tile_n
    )

    reference_points[
        "population_weight"
    ] = (
        reference_points[
            "tile_valid_pixels"
        ]
        / reference_points[
            "n_sample_in_tile"
        ]
    )

    parameter_rows = []

    for year in YEARS_TO_MAP:

        annual = pd.read_csv(
            annual_sample_path(
                year
            ),
            compression="gzip",
            usecols=(
                [
                    "point_id",
                    "valid_ia45",
                ]
                + IA_COLS
            ),
        )

        annual[
            "valid_ia45"
        ] = (
            annual[
                "valid_ia45"
            ]
            .astype(str)
            .str.lower()
            .map(
                {
                    "true": True,
                    "false": False,
                }
            )
        )

        merged = reference_points.merge(
            annual,
            on="point_id",
            how="left",
            validate="one_to_one",
        )

        if not merged[
            "valid_ia45"
        ].all():
            raise RuntimeError(
                f"Persistent-valid S500 support failed in {year}."
            )

        weights = merged[
            "population_weight"
        ].to_numpy(
            dtype=np.float64
        )

        for feature in IA_COLS:

            mu, sd = weighted_mean_sd(
                merged[
                    feature
                ].to_numpy(
                    dtype=np.float64
                ),
                weights,
            )

            parameter_rows.append(
                {
                    "sample_tier": "S500",
                    "tier_n": 500,
                    "Year": year,
                    "feature": feature,
                    "weighted_mean": mu,
                    "weighted_sd": sd,
                }
            )

    reference_parameters = pd.DataFrame(
        parameter_rows
    )

    reference_parameters.to_csv(
        OUTPUT_DIR
        / "annual_IA45_zscore_parameters_S500_reconstructed.csv",
        index=False,
    )


expected_parameter_rows = (
    len(
        YEARS_TO_MAP
    )
    * EXPECTED_IA_D
)

if len(
    reference_parameters
) != expected_parameter_rows:
    raise RuntimeError(
        f"Expected {expected_parameter_rows} z-score parameter rows; "
        f"found {len(reference_parameters)}."
    )


reference_lookup = {}

for year in YEARS_TO_MAP:

    year_params = (
        reference_parameters.loc[
            reference_parameters[
                "Year"
            ].eq(
                year
            )
        ]
        .set_index(
            "feature"
        )
    )

    reference_lookup[
        year
    ] = {
        "mean": np.array(
            [
                year_params.loc[
                    feature,
                    "weighted_mean",
                ]
                for feature in IA_COLS
            ],
            dtype=np.float64,
        ),
        "sd": np.array(
            [
                year_params.loc[
                    feature,
                    "weighted_sd",
                ]
                for feature in IA_COLS
            ],
            dtype=np.float64,
        ),
    }

print(
    "Annual z-score parameters ready."
)


Loaded existing S500 z-score parameters:
C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_IA45\annual_IA45_zscore_parameters_S500.csv
Annual z-score parameters ready.


## 1B. Recover POLARIS band names from VRT source shards

The 42-band VRT may not retain band descriptions even when the underlying GeoTIFF shards do.  
Before selecting any POLARIS predictors, this cell parses the VRT band-by-band, follows each
`SourceFilename` / `SourceBand` reference, and recovers the logical band name from the source
shard metadata. This works for both spatial shards that each contain all bands and one-band
shards used to construct a stacked VRT.

No band is accepted merely because of its physical position. If source metadata are absent,
the code can also recognize an explicit canonical POLARIS band token in the shard filename.
Any unresolved or conflicting VRT bands fail visibly before modeling.


In [14]:
# =============================================================================
# 1B. RECOVER POLARIS BAND SCHEMA FROM ONE AUTHORITATIVE SHARD
#
# The final exact mosaic:
#
#   C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_vrt\
#       POLARIS_26911_10m_mosaic_exact.tif
#
# has the correct 42-band order but no band descriptions.
#
# The original spatial shards DO retain those band descriptions.
# Because the mosaic was built by spatially mosaicking those shards without
# changing band order, read ONE complete shard and transfer its
# band-index -> band-name mapping to the exact mosaic.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio


# -----------------------------------------------------------------------------
# AUTHORITATIVE FINAL POLARIS RASTER
# -----------------------------------------------------------------------------

POLARIS_SOURCE = Path(
    r"C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_vrt"
    r"\POLARIS_26911_10m_mosaic_exact.tif"
)


if not POLARIS_SOURCE.exists():

    raise FileNotFoundError(
        f"POLARIS exact mosaic not found:\n"
        f"{POLARIS_SOURCE}"
    )


# -----------------------------------------------------------------------------
# DISCOVER ONE ORIGINAL SHARD
#
# Search recursively below the POLARIS directory.
# Exclude the final mosaic itself.
#
# We require:
#   - same number of bands as the exact mosaic
#   - every band has a description
#   - descriptions are unique
# -----------------------------------------------------------------------------

POLARIS_ROOT = Path(
    r"C:\NCA_DATA\Ancillary_Data\POLARIS"
)


with rasterio.open(
    POLARIS_SOURCE
) as mosaic_src:

    polaris_count = mosaic_src.count
    polaris_crs = mosaic_src.crs
    polaris_transform = mosaic_src.transform

    polaris_shape = (
        mosaic_src.height,
        mosaic_src.width,
    )


print(
    "Exact POLARIS mosaic:"
)

print(
    f"  {POLARIS_SOURCE}"
)

print(
    f"  bands: {polaris_count}"
)

print(
    f"  CRS: {polaris_crs}"
)

print(
    f"  shape: {polaris_shape}"
)


candidate_shards = sorted(
    [
        path
        for path in POLARIS_ROOT.rglob(
            "*.tif"
        )
        if path.resolve()
        != POLARIS_SOURCE.resolve()
    ]
)


if not candidate_shards:

    raise FileNotFoundError(
        f"No candidate POLARIS shard TIFFs found beneath:\n"
        f"{POLARIS_ROOT}"
    )


POLARIS_SCHEMA_SHARD = None
resolved_band_names = None


for shard_path in candidate_shards:

    try:

        with rasterio.open(
            shard_path
        ) as src:

            # Must represent the same 42-band stack.
            if src.count != polaris_count:
                continue


            names = [
                (
                    str(name).strip()
                    if (
                        name is not None
                        and str(name).strip()
                    )
                    else None
                )
                for name in src.descriptions
            ]


            # Require every band to have a name.
            if any(
                name is None
                for name in names
            ):
                continue


            # Require unique names.
            if len(
                set(
                    names
                )
            ) != len(
                names
            ):
                continue


            POLARIS_SCHEMA_SHARD = shard_path

            resolved_band_names = names

            break


    except Exception:

        continue


if POLARIS_SCHEMA_SHARD is None:

    raise RuntimeError(
        "Could not find a complete POLARIS shard with "
        f"{polaris_count} named bands beneath:\n"
        f"{POLARIS_ROOT}"
    )


print(
    "\nAuthoritative band schema read from:"
)

print(
    f"  {POLARIS_SCHEMA_SHARD}"
)


# -----------------------------------------------------------------------------
# CREATE BAND INDEX -> NAME TABLE FOR THE EXACT MOSAIC
# -----------------------------------------------------------------------------

POLARIS_ALL_BANDS = pd.DataFrame(
    {
        "band_index": np.arange(
            1,
            polaris_count + 1,
            dtype=int,
        ),

        "band_name": resolved_band_names,
    }
)


print(
    "\nTransferred shard schema to exact mosaic band indexes:"
)

display(
    POLARIS_ALL_BANDS
)


# Strong QA:
# we expect the current POLARIS stack to contain exactly 42 bands.
if polaris_count != 42:

    raise ValueError(
        f"Expected 42 POLARIS bands, "
        f"but exact mosaic contains {polaris_count}."
    )


POLARIS_ALL_BANDS.to_csv(
    OUTPUT_DIR
    / "POLARIS_exact_mosaic_band_schema_from_shard.csv",
    index=False,
)

Exact POLARIS mosaic:
  C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif
  bands: 42
  CRS: PROJCS["NAD83 / UTM zone 11N",GEOGCS["NAD83",DATUM["North American Datum 1983",SPHEROID["GRS 1980",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
  shape: (7857, 10435)

Authoritative band schema read from:
  C:\NCA_DATA\Ancillary_Data\POLARIS\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000000000-0000000000.tif

Transferred shard schema to exact mosaic band indexes:


,band_index,band_name
0,1,sand_0_5
1,2,sand_5_15
2,3,sand_15_30
3,4,sand_30_60
4,5,sand_60_100
5,6,sand_100_200
6,7,silt_0_5
7,8,silt_5_15
8,9,silt_15_30
9,10,silt_30_60


## 2. Select the 0–30 cm POLARIS source bands

Band identity has already been recovered from the VRT's actual source shards in Section 1B.
This section selects the requested seven variables at 0–5, 5–15, and 15–30 cm by semantic
name and constructs the 21-band read index used for the depth-weighted 0–30 cm aggregation.


In [15]:
# =============================================================================
# 2. SELECT POLARIS 0–30 CM SOURCE BANDS
#
# Seven soil variables:
#   sand
#   silt
#   clay
#   bd
#   theta_s
#   ph
#   om_log10
#
# Three source depths:
#   0–5 cm
#   5–15 cm
#   15–30 cm
#
# These 21 source bands will later be condensed to seven 0–30 cm predictors.
# =============================================================================

POLARIS_VARIABLES = [
    "sand",
    "silt",
    "clay",
    "bd",
    "theta_s",
    "ph",
    "om_log10",
]


POLARIS_DEPTHS = [
    "0_5",
    "5_15",
    "15_30",
]


def normalize_name(
    value,
):

    return (
        str(
            value
        )
        .strip()
        .lower()
    )


POLARIS_ALL_BANDS[
    "normalized_name"
] = POLARIS_ALL_BANDS[
    "band_name"
].map(
    normalize_name
)


resolved_rows = []


for variable in POLARIS_VARIABLES:

    for depth in POLARIS_DEPTHS:

        expected_name = (
            f"{variable}_{depth}"
        )

        matches = POLARIS_ALL_BANDS.loc[
            POLARIS_ALL_BANDS[
                "normalized_name"
            ].eq(
                expected_name.lower()
            )
        ].copy()


        if len(
            matches
        ) == 0:

            raise KeyError(
                f"POLARIS band not found: "
                f"{expected_name}\n\n"
                f"Available bands:\n"
                f"{POLARIS_ALL_BANDS['band_name'].tolist()}"
            )


        if len(
            matches
        ) > 1:

            raise ValueError(
                f"Duplicate POLARIS band match "
                f"for {expected_name}:\n"
                f"{matches}"
            )


        match = matches.iloc[
            0
        ]


        resolved_rows.append(
            {
                "variable": variable,
                "depth": depth,

                "band_index": int(
                    match[
                        "band_index"
                    ]
                ),

                "band_name": str(
                    match[
                        "band_name"
                    ]
                ),
            }
        )


POLARIS_SELECTED_BAND_QA = pd.DataFrame(
    resolved_rows
)


SELECTED_POLARIS_INDEXES = (
    POLARIS_SELECTED_BAND_QA[
        "band_index"
    ]
    .astype(
        int
    )
    .tolist()
)


SELECTED_POLARIS_BANDS = (
    POLARIS_SELECTED_BAND_QA[
        "band_name"
    ]
    .astype(
        str
    )
    .tolist()
)


if len(
    SELECTED_POLARIS_INDEXES
) != 21:

    raise RuntimeError(
        f"Expected 21 source bands "
        f"(7 variables × 3 depths); "
        f"found {len(SELECTED_POLARIS_INDEXES)}."
    )


print(
    "Selected POLARIS bands for 0–30 cm aggregation:"
)

display(
    POLARIS_SELECTED_BAND_QA
)


POLARIS_SELECTED_BAND_QA.to_csv(
    OUTPUT_DIR
    / "POLARIS_selected_0_30_source_bands.csv",
    index=False,
)

Selected POLARIS bands for 0–30 cm aggregation:


,variable,depth,band_index,band_name
0,sand,0_5,1,sand_0_5
1,sand,5_15,2,sand_5_15
2,sand,15_30,3,sand_15_30
3,silt,0_5,7,silt_0_5
4,silt,5_15,8,silt_5_15
5,silt,15_30,9,silt_15_30
6,clay,0_5,13,clay_0_5
7,clay,5_15,14,clay_5_15
8,clay,15_30,15,clay_15_30
9,bd,0_5,19,bd_0_5


In [16]:
# =============================================================================
# 2B. DEPTH-WEIGHTED 0–30 CM POLARIS AGGREGATION
#
# Input:
#   21 source bands
#   = 7 variables × 3 depths
#
# Output:
#   7 predictors representing the upper 0–30 cm profile
#
# Depth weights:
#   0–5 cm   = 5 / 30
#   5–15 cm  = 10 / 30
#   15–30 cm = 15 / 30
# =============================================================================


# -----------------------------------------------------------------------------
# DEPTH THICKNESSES
# -----------------------------------------------------------------------------

POLARIS_DEPTH_THICKNESS_CM = {
    "0_5": 5.0,
    "5_15": 10.0,
    "15_30": 15.0,
}


# -----------------------------------------------------------------------------
# OUTPUT FEATURE NAMES
# -----------------------------------------------------------------------------

POLARIS_AGG_FEATURE_NAMES = [
    f"{variable}_0_30_dw"
    for variable in POLARIS_VARIABLES
]


# -----------------------------------------------------------------------------
# MAP VARIABLE + DEPTH -> POSITION IN THE 21-BAND EXTRACTED ARRAY
#
# IMPORTANT:
# These are positions in SELECTED_POLARIS_INDEXES,
# NOT physical raster band numbers.
# -----------------------------------------------------------------------------

POLARIS_SELECTED_POSITION_LOOKUP = {
    (
        str(row.variable),
        str(row.depth),
    ): int(i)

    for i, row in (
        POLARIS_SELECTED_BAND_QA
        .reset_index(
            drop=True
        )
        .iterrows()
    )
}


# -----------------------------------------------------------------------------
# NORMALIZED THICKNESS WEIGHTS
# -----------------------------------------------------------------------------

POLARIS_DEPTH_WEIGHT_VECTOR = np.array(
    [
        POLARIS_DEPTH_THICKNESS_CM[
            depth
        ]
        for depth in POLARIS_DEPTHS
    ],
    dtype=np.float64,
)


POLARIS_DEPTH_WEIGHT_VECTOR = (
    POLARIS_DEPTH_WEIGHT_VECTOR
    / POLARIS_DEPTH_WEIGHT_VECTOR.sum()
)


print(
    "POLARIS 0–30 cm weights:"
)

for depth, weight in zip(
    POLARIS_DEPTHS,
    POLARIS_DEPTH_WEIGHT_VECTOR,
):

    print(
        f"  {depth:8s}: "
        f"{weight:.4f}"
    )


# =============================================================================
# AGGREGATION FUNCTION
# =============================================================================

def aggregate_polaris_0_30(
    raw_selected,
):
    """
    Collapse the 21 selected POLARIS bands
    (7 variables × 3 depths)
    to seven depth-weighted 0–30 cm predictors.

    Parameters
    ----------
    raw_selected : ndarray
        Shape (..., 21).

        Last dimension must follow the order in
        POLARIS_SELECTED_BAND_QA / SELECTED_POLARIS_INDEXES.

    Returns
    -------
    ndarray
        Shape (..., 7), ordered according to POLARIS_VARIABLES.

    Rules
    -----
    sand, silt, clay, bd, theta_s, ph:
        thickness-weighted arithmetic profile summary.

    om_log10:
        back-transform each depth to linear OM,
        calculate thickness-weighted arithmetic mean,
        then return to log10 space.
    """

    raw_selected = np.asarray(
        raw_selected,
        dtype=np.float64,
    )


    expected_features = len(
        SELECTED_POLARIS_INDEXES
    )


    if (
        raw_selected.shape[-1]
        != expected_features
    ):

        raise ValueError(
            "POLARIS feature dimension mismatch: "
            f"expected {expected_features}, "
            f"received {raw_selected.shape[-1]}."
        )


    aggregated = []


    for variable in POLARIS_VARIABLES:

        # Positions of this variable's:
        #   0–5
        #   5–15
        #   15–30 cm
        positions = [
            POLARIS_SELECTED_POSITION_LOOKUP[
                (
                    variable,
                    depth,
                )
            ]
            for depth in POLARIS_DEPTHS
        ]


        values = raw_selected[
            ...,
            positions
        ]


        # ---------------------------------------------------------------------
        # ORGANIC MATTER
        #
        # Stored as log10(OM), so do not average the logged values directly.
        # ---------------------------------------------------------------------

        if variable == "om_log10":

            linear_om = np.power(
                10.0,
                values,
            )


            weighted_linear_om = np.sum(
                linear_om
                * POLARIS_DEPTH_WEIGHT_VECTOR,
                axis=-1,
            )


            aggregate = np.log10(
                np.maximum(
                    weighted_linear_om,
                    np.finfo(
                        np.float64
                    ).tiny,
                )
            )


        # ---------------------------------------------------------------------
        # OTHER VARIABLES
        # ---------------------------------------------------------------------

        else:

            aggregate = np.sum(
                values
                * POLARIS_DEPTH_WEIGHT_VECTOR,
                axis=-1,
            )


        aggregated.append(
            aggregate
        )


    return np.stack(
        aggregated,
        axis=-1,
    )


# =============================================================================
# QA
# =============================================================================

print(
    "\nRaw POLARIS source dimensions:",
    len(
        SELECTED_POLARIS_INDEXES
    ),
)

print(
    "Aggregated POLARIS dimensions:",
    len(
        POLARIS_AGG_FEATURE_NAMES
    ),
)

print(
    "\nAggregated features:"
)

for feature in POLARIS_AGG_FEATURE_NAMES:

    print(
        " ",
        feature,
    )


# Check that all 7 × 3 variable/depth combinations exist.
expected_combinations = {
    (
        variable,
        depth,
    )
    for variable in POLARIS_VARIABLES
    for depth in POLARIS_DEPTHS
}

actual_combinations = set(
    POLARIS_SELECTED_POSITION_LOOKUP.keys()
)


if (
    actual_combinations
    != expected_combinations
):

    missing = sorted(
        expected_combinations
        - actual_combinations
    )

    extra = sorted(
        actual_combinations
        - expected_combinations
    )

    raise ValueError(
        "POLARIS aggregation lookup is incomplete.\n"
        f"Missing: {missing}\n"
        f"Extra: {extra}"
    )


print(
    "\nPOLARIS aggregation QA passed."
)

POLARIS 0–30 cm weights:
  0_5     : 0.1667
  5_15    : 0.3333
  15_30   : 0.5000

Raw POLARIS source dimensions: 21
Aggregated POLARIS dimensions: 7

Aggregated features:
  sand_0_30_dw
  silt_0_30_dw
  clay_0_30_dw
  bd_0_30_dw
  theta_s_0_30_dw
  ph_0_30_dw
  om_log10_0_30_dw

POLARIS aggregation QA passed.


## 3. Load vegetation training observations and align IA45 features

For a fair representation comparison, the final analysis dataset uses the
**same complete-case rows for all three feature sets**. If a vegetation point
does not have valid selected POLARIS values, that row is removed from Z-only,
POLARIS-only, and Z+POLARIS alike.


In [17]:
# =============================================================================
# 3. LOAD VEGETATION TRAINING TABLE
# =============================================================================

veg = pd.read_csv(
    VEG_TRAINING_CSV,
    low_memory=False,
).drop(
    columns=[
        "Unnamed: 0",
    ],
    errors="ignore",
)

veg[
    "Year"
] = pd.to_numeric(
    veg[
        "Year"
    ],
    errors="coerce",
)

veg[
    "parent_cluster"
] = pd.to_numeric(
    veg[
        "parent_cluster"
    ],
    errors="coerce",
)

veg = (
    veg.loc[
        veg[
            "Year"
        ].isin(
            VEG_YEARS
        )
        & veg[
            "parent_cluster"
        ].isin(
            VEG_PARENT_CODES
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

veg[
    "Year"
] = veg[
    "Year"
].astype(
    int
)

veg[
    "parent_cluster"
] = veg[
    "parent_cluster"
].astype(
    int
)

veg[
    "PlotID"
] = (
    veg[
        "PlotID"
    ]
    .astype(str)
    .str.strip()
)


def normalize_feature_name(
    x,
):
    return (
        str(
            x
        )
        .strip()
        .lower()
    )


veg_candidate_cols = []

for col in veg.columns:

    lower = normalize_feature_name(
        col
    )

    is_intercept = (
        re.search(
            r"(^|_)(intercept|constant)($|_)",
            lower,
        )
        is not None
    )

    is_amplitude = (
        re.search(
            r"(^|_)(amp|amplitude)",
            lower,
        )
        is not None
    )

    if (
        is_intercept
        or is_amplitude
    ):
        veg_candidate_cols.append(
            col
        )


if len(
    veg_candidate_cols
) != EXPECTED_IA_D:
    raise ValueError(
        f"Expected {EXPECTED_IA_D} IA45 columns in vegetation table; "
        f"found {len(veg_candidate_cols)}."
    )


sample_lookup = {
    normalize_feature_name(
        c
    ): c
    for c in IA_COLS
}

veg_to_sample = {}

for veg_col in veg_candidate_cols:

    key = normalize_feature_name(
        veg_col
    )

    if key not in sample_lookup:
        raise KeyError(
            f"Vegetation IA45 feature does not match optical sample: {veg_col}"
        )

    veg_to_sample[
        veg_col
    ] = sample_lookup[
        key
    ]


sample_to_veg = {
    sample_name: veg_name
    for veg_name, sample_name in veg_to_sample.items()
}

VEG_IA_COLS = [
    sample_to_veg[
        feature
    ]
    for feature in IA_COLS
]


required_xy = [
    "Easting",
    "Northing",
]

if not all(
    c in veg.columns
    for c in required_xy
):
    raise KeyError(
        "Vegetation table must contain Easting and Northing."
    )

print(
    "Vegetation rows before POLARIS complete-case filtering:",
    len(
        veg
    )
)


Vegetation rows before POLARIS complete-case filtering: 479


## 4. Extract and depth-integrate POLARIS predictors at vegetation plots

The 21 required POLARIS source bands are sampled on the Sentinel grid through a
`WarpedVRT` when necessary. The three upper-soil depth intervals are then collapsed to
seven thickness-weighted `0–30 cm` predictors using the same aggregation function that
will later be applied blockwise during raster mapping.

This keeps training and wall-to-wall prediction feature construction identical.


In [18]:
# =============================================================================
# 4. EXTRACT + AGGREGATE POLARIS AT VEGETATION POINTS
# =============================================================================

reference_raster_path = annual_k2_path(
    YEARS_TO_MAP[0]
)

with rasterio.open(
    reference_raster_path
) as ref:

    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height


def grids_match_definition(
    src,
    crs,
    transform,
    width,
    height,
):
    return (
        src.crs == crs
        and src.transform == transform
        and src.width == width
        and src.height == height
    )


coords = list(
    zip(
        veg[
            "Easting"
        ].astype(
            float
        ),
        veg[
            "Northing"
        ].astype(
            float
        ),
    )
)


with rasterio.open(
    POLARIS_SOURCE
) as polaris_src:

    if grids_match_definition(
        polaris_src,
        ref_crs,
        ref_transform,
        ref_width,
        ref_height,
    ):

        polaris_reader = polaris_src
        polaris_vrt = None

    else:

        polaris_vrt = WarpedVRT(
            polaris_src,
            crs=ref_crs,
            transform=ref_transform,
            width=ref_width,
            height=ref_height,
            resampling=POLARIS_RESAMPLING,
        )

        polaris_reader = polaris_vrt


    sampled_rows = []

    for sample in polaris_reader.sample(
        coords,
        indexes=SELECTED_POLARIS_INDEXES,
        masked=True,
    ):

        sampled_rows.append(
            np.asarray(
                sample.filled(
                    np.nan
                ),
                dtype=np.float64,
            )
        )


    if polaris_vrt is not None:
        polaris_vrt.close()


X_polaris_raw_all = np.vstack(
    sampled_rows
)

X_polaris_agg_all = aggregate_polaris_0_30(
    X_polaris_raw_all
)


for j, feature in enumerate(
    POLARIS_AGG_FEATURE_NAMES
):

    veg[
        f"POLARIS__{feature}"
    ] = X_polaris_agg_all[
        :,
        j
    ]


valid_polaris = np.isfinite(
    X_polaris_agg_all
).all(
    axis=1
)


print(
    "Rows with complete aggregated POLARIS predictors:",
    f"{valid_polaris.sum():,} / {len(veg):,}"
)


# Optional mask filter retained from prior notebooks.
if FILTER_TRAINING_TO_MASK:

    def keep_mask_at_points(
        table,
    ):

        with rasterio.open(
            KEEP_MASK
        ) as src:

            sampled = list(
                src.sample(
                    list(
                        zip(
                            table[
                                "Easting"
                            ].astype(
                                float
                            ),
                            table[
                                "Northing"
                            ].astype(
                                float
                            ),
                        )
                    ),
                    indexes=1,
                    masked=True,
                )
            )

        keep = np.array(
            [
                (
                    not s.mask[
                        0
                    ]
                    and np.isfinite(
                        s[
                            0
                        ]
                    )
                    and s[
                        0
                    ] > 0
                )
                for s in sampled
            ],
            dtype=bool,
        )

        return keep


    valid_mask = keep_mask_at_points(
        veg
    )

else:

    valid_mask = np.ones(
        len(
            veg
        ),
        dtype=bool,
    )


analysis_keep = (
    valid_polaris
    & valid_mask
)


veg = (
    veg.loc[
        analysis_keep
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


X_polaris = (
    veg[
        [
            f"POLARIS__{feature}"
            for feature in POLARIS_AGG_FEATURE_NAMES
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


print(
    "Final common analysis rows:",
    len(
        veg
    )
)

print(
    "Aggregated POLARIS dimensions:",
    X_polaris.shape[
        1
    ],
)

display(
    veg[
        [
            "PlotID",
            "Year",
            "parent_cluster",
        ]
        + [
            f"POLARIS__{feature}"
            for feature in POLARIS_AGG_FEATURE_NAMES
        ]
    ].head()
)


Rows with complete aggregated POLARIS predictors: 479 / 479
Final common analysis rows: 479
Aggregated POLARIS dimensions: 7


,PlotID,Year,parent_cluster,POLARIS__sand_0_30_dw,POLARIS__silt_0_30_dw,POLARIS__clay_0_30_dw,POLARIS__bd_0_30_dw,POLARIS__theta_s_0_30_dw,POLARIS__ph_0_30_dw,POLARIS__om_log10_0_30_dw
0,100,2016,3,22.077781,64.685916,11.591524,1.419252,0.464433,7.780439,0.102986
1,101,2016,3,22.045268,64.872700,11.594889,1.420790,0.463853,7.773752,0.103538
2,102,2016,4,46.500000,34.042500,16.166667,1.303333,0.508176,7.601570,-0.154604
3,103,2016,1,46.500000,33.076246,16.166667,1.303333,0.508176,7.690096,-0.153672
4,104,2016,1,46.500000,38.065776,16.166667,1.303333,0.508176,7.568698,-0.152111


## 5. Build the three feature representations

The same final rows, labels, groups, and CV folds are used for:

- `z`: 45 annual optical z-score features;
- `polaris`: 7 depth-integrated static POLARIS features;
- `z_plus_polaris`: 52 total predictors (45 optical + 7 POLARIS).

This retains the controlled feature-domain comparison while replacing the previous
18-dimensional depth-explicit texture representation with the new 7-dimensional
integrated edaphic representation.


In [19]:
# =============================================================================
# 5. BUILD FEATURE MATRICES
# =============================================================================

X_raw_ia = (
    veg[
        VEG_IA_COLS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    X_raw_ia
).all():
    raise ValueError(
        "Invalid IA45 values remain in the common analysis dataset."
    )


years_veg = veg[
    "Year"
].to_numpy(
    dtype=int
)

X_z = np.full_like(
    X_raw_ia,
    np.nan,
    dtype=np.float64,
)


for year in np.unique(
    years_veg
):

    mask = (
        years_veg
        == year
    )

    params = reference_lookup[
        int(
            year
        )
    ]

    X_z[
        mask
    ] = (
        X_raw_ia[
            mask
        ]
        - params[
            "mean"
        ]
    ) / params[
        "sd"
    ]


if not np.isfinite(
    X_z
).all():
    raise ValueError(
        "Annual z-score transformation produced invalid values."
    )


X_z_plus_polaris = np.hstack(
    [
        X_z,
        X_polaris,
    ]
)


REPRESENTATIONS = {
    "z": X_z,
    "polaris": X_polaris,
    "z_plus_polaris": X_z_plus_polaris,
}


REPRESENTATION_FEATURE_NAMES = {
    "z": [
        f"z__{feature}"
        for feature in IA_COLS
    ],
    "polaris": [
        f"POLARIS__{feature}"
        for feature in POLARIS_AGG_FEATURE_NAMES
    ],
    "z_plus_polaris": (
        [
            f"z__{feature}"
            for feature in IA_COLS
        ]
        + [
            f"POLARIS__{feature}"
            for feature in POLARIS_AGG_FEATURE_NAMES
        ]
    ),
}


representation_qa = pd.DataFrame(
    [
        {
            "representation": name,
            "n_rows": X.shape[
                0
            ],
            "n_features": X.shape[
                1
            ],
        }
        for name, X in REPRESENTATIONS.items()
    ]
)

display(
    representation_qa
)


# Save the explicit common ML dataset.
ml_dataset = veg[
    [
        "PlotID",
        "Year",
        "parent_cluster",
        "Easting",
        "Northing",
    ]
].copy()

for j, feature in enumerate(
    REPRESENTATION_FEATURE_NAMES[
        "z"
    ]
):
    ml_dataset[
        feature
    ] = X_z[
        :,
        j
    ]

for j, feature in enumerate(
    REPRESENTATION_FEATURE_NAMES[
        "polaris"
    ]
):
    ml_dataset[
        feature
    ] = X_polaris[
        :,
        j
    ]

ml_dataset.to_csv(
    OUTPUT_DIR
    / "FieldVeg_AnnualZScore_POLARIS_common_ML_dataset.csv",
    index=False,
)


,representation,n_rows,n_features
0,z,479,45
1,polaris,479,7
2,z_plus_polaris,479,52


## 6. Models and fixed grouped folds

The fold assignment is generated once and reused across all three
representations.


In [20]:
# =============================================================================
# 6. MODELS
# =============================================================================

MODELS = {
    "elastic_net_logistic": Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                solver="saga",
                penalty="elasticnet",
                l1_ratio=0.5,
                C=1.0,
                class_weight="balanced",
                max_iter=10000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=800,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        bootstrap=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "extra_trees": ExtraTreesClassifier(
        n_estimators=800,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        bootstrap=False,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}


y = (
    "VEG::"
    + veg[
        "parent_cluster"
    ].astype(
        int
    ).astype(
        str
    )
).to_numpy()

groups = veg[
    "PlotID"
].astype(
    str
).to_numpy()


cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


# Same folds for every representation.
CV_SPLITS = list(
    cv.split(
        X_z,
        y,
        groups,
    )
)


print(
    "Common CV folds:",
    len(
        CV_SPLITS
    )
)


Common CV folds: 5


## 7. Grouped OOF comparison


In [21]:
# =============================================================================
# 7. GROUPED OUT-OF-FOLD COMPARISON
# =============================================================================

def score_predictions(
    y_true,
    y_pred,
):

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=VEG_CLASSES,
            average="macro",
            zero_division=0,
        ),
    }


oof_fold_records = []
oof_predictions = {}


for representation, X in REPRESENTATIONS.items():

    oof_predictions[
        representation
    ] = {}

    for model_name, template in MODELS.items():

        pred_all = np.empty(
            len(
                y
            ),
            dtype=object,
        )

        pred_all[:] = None

        for fold, (
            train_idx,
            test_idx,
        ) in enumerate(
            CV_SPLITS,
            start=1,
        ):

            fitted = clone(
                template
            )

            fitted.fit(
                X[
                    train_idx
                ],
                y[
                    train_idx
                ],
            )

            pred = fitted.predict(
                X[
                    test_idx
                ]
            ).astype(
                str
            )

            pred_all[
                test_idx
            ] = pred

            fold_scores = score_predictions(
                y[
                    test_idx
                ],
                pred,
            )

            oof_fold_records.append(
                {
                    "representation": representation,
                    "model": model_name,
                    "fold": fold,
                    "n_test": len(
                        test_idx
                    ),
                    **fold_scores,
                }
            )

        oof_predictions[
            representation
        ][
            model_name
        ] = pred_all


oof_fold_df = pd.DataFrame(
    oof_fold_records
)

oof_fold_df.to_csv(
    OUTPUT_DIR
    / "representation_groupedOOF_fold_metrics.csv",
    index=False,
)


oof_fold_summary = (
    oof_fold_df.groupby(
        [
            "representation",
            "model",
        ],
        as_index=False,
    )
    .agg(
        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean",
        ),
        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        macro_f1_sd=(
            "macro_f1",
            "std",
        ),
    )
    .sort_values(
        [
            "representation",
            "macro_f1_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(
    oof_fold_summary.round(
        4
    )
)


,representation,model,accuracy_mean,balanced_accuracy_mean,macro_f1_mean,macro_f1_sd
1,polaris,extra_trees,0.2501,0.2341,0.2261,0.0562
2,polaris,random_forest,0.2439,0.2241,0.2143,0.0559
0,polaris,elastic_net_logistic,0.1770,0.1777,0.1522,0.0375
4,z,extra_trees,0.4947,0.4480,0.4417,0.0237
3,z,elastic_net_logistic,0.4658,0.4690,0.4388,0.0880
5,z,random_forest,0.4864,0.4260,0.4269,0.0308
6,z_plus_polaris,elastic_net_logistic,0.4949,0.4883,0.4629,0.0585
7,z_plus_polaris,extra_trees,0.5053,0.4525,0.4486,0.0361
8,z_plus_polaris,random_forest,0.5095,0.4458,0.4450,0.0340


## 8. Pooled grouped-OOF metrics and mapping-model confusion matrices


In [22]:
# =============================================================================
# 8. POOLED OOF
# =============================================================================

pooled_records = []

for representation in REPRESENTATIONS:

    for model_name in MODELS:

        pred = np.asarray(
            oof_predictions[
                representation
            ][
                model_name
            ]
        ).astype(
            str
        )

        pooled_records.append(
            {
                "representation": representation,
                "model": model_name,
                **score_predictions(
                    y,
                    pred,
                ),
            }
        )


pooled_metrics = pd.DataFrame(
    pooled_records
)

pooled_metrics.to_csv(
    OUTPUT_DIR
    / "representation_groupedOOF_pooled_metrics.csv",
    index=False,
)

display(
    pooled_metrics.sort_values(
        [
            "model",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    ).round(
        4
    )
)


print(
    "\nOOF confusion matrices for mapping model:",
    MODEL_FOR_MAPPING,
)


for representation in REPRESENTATIONS:

    pred = np.asarray(
        oof_predictions[
            representation
        ][
            MODEL_FOR_MAPPING
        ]
    ).astype(
        str
    )

    cm = confusion_matrix(
        y,
        pred,
        labels=VEG_CLASSES,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"True P{i}"
            for i in VEG_PARENT_CODES
        ],
        columns=[
            f"Pred P{i}"
            for i in VEG_PARENT_CODES
        ],
    )

    print(
        "\n",
        representation,
    )

    display(
        cm_df
    )

    cm_df.to_csv(
        OUTPUT_DIR
        / (
            f"OOF_confusion_{representation}_"
            f"{MODEL_FOR_MAPPING}.csv"
        )
    )


,representation,model,accuracy,balanced_accuracy,macro_f1
6,z_plus_polaris,elastic_net_logistic,0.4948,0.4956,0.4766
0,z,elastic_net_logistic,0.4656,0.4779,0.4536
3,polaris,elastic_net_logistic,0.1775,0.1763,0.1587
8,z_plus_polaris,extra_trees,0.5052,0.4551,0.4626
2,z,extra_trees,0.4948,0.4556,0.4618
5,polaris,extra_trees,0.2505,0.2365,0.2338
7,z_plus_polaris,random_forest,0.5094,0.4530,0.4620
1,z,random_forest,0.4864,0.4362,0.4469
4,polaris,random_forest,0.2443,0.2214,0.2246



OOF confusion matrices for mapping model: random_forest

 z


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,47,4,3,9,9,7,4,0
True P2,9,27,12,1,5,0,4,1
True P3,3,16,43,3,11,0,2,0
True P4,8,0,6,52,4,11,0,1
True P5,8,5,17,4,33,6,1,0
True P6,4,3,3,16,7,18,0,0
True P7,10,9,2,2,0,0,9,0
True P8,0,0,5,5,5,1,0,4



 polaris


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,22,11,9,15,12,5,7,2
True P2,11,14,9,7,8,1,6,3
True P3,13,8,22,13,8,2,8,4
True P4,14,3,10,26,9,13,7,0
True P5,12,7,6,10,17,9,12,1
True P6,12,0,1,15,8,10,3,2
True P7,8,2,5,2,8,2,3,2
True P8,3,7,3,1,1,1,1,3



 z_plus_polaris


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,50,4,4,8,6,7,4,0
True P2,6,29,12,2,4,0,5,1
True P3,3,14,44,3,12,0,2,0
True P4,7,0,7,55,3,9,0,1
True P5,7,4,15,3,37,7,0,1
True P6,5,1,4,17,8,16,0,0
True P7,10,8,3,2,0,0,9,0
True P8,0,0,5,4,5,2,0,4


## 9. Leave-one-year-out comparison

This is the principal temporal-transfer test.

For `Z + POLARIS`, target-year optical normalization remains unlabeled, while
the POLARIS predictors are static covariates observed everywhere.

Note that a static soil domain can make repeated locations easier to recognize.
LOYO therefore remains directly comparable to the current baseline, but a
later spatially blocked temporal test should be treated as the stronger
generalization test once soils are incorporated.


In [23]:
# =============================================================================
# 9. LEAVE-ONE-YEAR-OUT
# =============================================================================

loyo_records = []
loyo_predictions = {}


for representation, X in REPRESENTATIONS.items():

    loyo_predictions[
        representation
    ] = {}

    for model_name, template in MODELS.items():

        loyo_predictions[
            representation
        ][
            model_name
        ] = {}

        for held_year in VEG_YEARS:

            train_mask = (
                years_veg
                != held_year
            )

            test_mask = (
                years_veg
                == held_year
            )

            fitted = clone(
                template
            )

            fitted.fit(
                X[
                    train_mask
                ],
                y[
                    train_mask
                ],
            )

            pred = fitted.predict(
                X[
                    test_mask
                ]
            ).astype(
                str
            )

            loyo_predictions[
                representation
            ][
                model_name
            ][
                held_year
            ] = pred

            loyo_records.append(
                {
                    "representation": representation,
                    "model": model_name,
                    "held_out_year": held_year,
                    "n_train": int(
                        train_mask.sum()
                    ),
                    "n_test": int(
                        test_mask.sum()
                    ),
                    **score_predictions(
                        y[
                            test_mask
                        ],
                        pred,
                    ),
                }
            )


loyo_results = pd.DataFrame(
    loyo_records
)

loyo_results.to_csv(
    OUTPUT_DIR
    / "representation_leaveYearOut.csv",
    index=False,
)

display(
    loyo_results.sort_values(
        [
            "held_out_year",
            "model",
            "macro_f1",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    ).round(
        4
    )
)


,representation,model,held_out_year,n_train,n_test,accuracy,balanced_accuracy,macro_f1
0,z,elastic_net_logistic,2016,224,255,0.3255,0.3147,0.3089
18,z_plus_polaris,elastic_net_logistic,2016,224,255,0.3176,0.3112,0.3020
9,polaris,elastic_net_logistic,2016,224,255,0.1451,0.1634,0.1455
24,z_plus_polaris,extra_trees,2016,224,255,0.3647,0.3275,0.3285
6,z,extra_trees,2016,224,255,0.3490,0.3042,0.3052
15,polaris,extra_trees,2016,224,255,0.1569,0.1484,0.1442
21,z_plus_polaris,random_forest,2016,224,255,0.3529,0.3073,0.3149
3,z,random_forest,2016,224,255,0.3412,0.2848,0.2792
12,polaris,random_forest,2016,224,255,0.1725,0.1694,0.1696
1,z,elastic_net_logistic,2018,373,106,0.3679,0.4519,0.3417


In [24]:
# =============================================================================
# 9B. LOYO CONFUSION MATRICES — MAPPING MODEL
# =============================================================================

for representation in REPRESENTATIONS:

    for held_year in VEG_YEARS:

        test_mask = (
            years_veg
            == held_year
        )

        pred = loyo_predictions[
            representation
        ][
            MODEL_FOR_MAPPING
        ][
            held_year
        ]

        cm = confusion_matrix(
            y[
                test_mask
            ],
            pred,
            labels=VEG_CLASSES,
        )

        cm_df = pd.DataFrame(
            cm,
            index=[
                f"True P{i}"
                for i in VEG_PARENT_CODES
            ],
            columns=[
                f"Pred P{i}"
                for i in VEG_PARENT_CODES
            ],
        )

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"{representation} | "
            f"{MODEL_FOR_MAPPING} | "
            f"held out {held_year}"
        )

        print(
            "=" * 80
        )

        display(
            cm_df
        )

        cm_df.to_csv(
            OUTPUT_DIR
            / (
                f"LOYO_confusion_{representation}_"
                f"{MODEL_FOR_MAPPING}_{held_year}.csv"
            )
        )



z | random_forest | held out 2016


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,17,1,7,4,4,3,2,4
True P2,5,2,14,1,3,1,1,1
True P3,4,1,21,6,8,2,0,0
True P4,5,0,3,20,6,3,1,5
True P5,8,2,14,1,18,2,0,1
True P6,2,0,3,11,2,6,0,0
True P7,8,1,4,2,0,0,2,2
True P8,0,0,2,3,5,0,0,1



z | random_forest | held out 2018


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,2,0,2,5,0,1,1,0
True P2,4,3,0,0,3,0,7,0
True P3,1,6,6,2,8,0,3,1
True P4,1,0,4,4,0,0,0,0
True P5,2,2,2,2,14,2,0,0
True P6,2,0,2,5,1,2,0,0
True P7,1,2,0,0,0,0,2,0
True P8,0,0,0,0,0,1,0,0



z | random_forest | held out 2025


...

,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,16,1,7,4,4,4,2,4
True P2,3,6,10,2,5,0,1,1
True P3,2,1,22,8,8,1,0,0
True P4,7,0,3,18,6,4,0,5
True P5,9,2,15,1,17,2,0,0
True P6,3,1,2,11,1,6,0,0
True P7,7,1,3,2,0,0,4,2
True P8,0,0,2,3,5,0,0,1



z_plus_polaris | random_forest | held out 2018


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,1,0,1,7,1,1,0,0
True P2,4,3,0,0,2,0,8,0
True P3,1,6,9,1,6,0,3,1
True P4,1,0,4,4,0,0,0,0
True P5,2,2,3,2,13,2,0,0
True P6,1,0,2,5,1,3,0,0
True P7,0,2,0,1,0,0,2,0
True P8,0,0,0,0,0,1,0,0



z_plus_polaris | random_forest | held out 2025


,Pred P1,Pred P2,Pred P3,Pred P4,Pred P5,Pred P6,Pred P7,Pred P8
True P1,10,4,1,2,9,2,2,0
True P2,1,5,6,0,1,0,1,0
True P3,1,7,1,0,0,0,0,0
True P4,0,0,5,8,0,16,0,1
True P5,0,0,0,0,4,0,0,0
True P6,0,0,0,1,7,7,0,0
True P7,0,2,1,0,0,0,5,0
True P8,0,3,1,1,2,1,0,0


## 10. Fit final mapping models and inspect soil feature importance

The same mapping learner is fit separately to each representation.


In [25]:
# =============================================================================
# 10. FINAL MAPPING MODELS
# =============================================================================

MAPPING_MODELS = {}

for representation, X in REPRESENTATIONS.items():

    fitted = clone(
        MODELS[
            MODEL_FOR_MAPPING
        ]
    )

    fitted.fit(
        X,
        y,
    )

    MAPPING_MODELS[
        representation
    ] = fitted

    print(
        representation,
        "| fitted rows:",
        len(
            y
        ),
        "| features:",
        X.shape[
            1
        ],
    )


z | fitted rows: 479 | features: 45
polaris | fitted rows: 479 | features: 7
z_plus_polaris | fitted rows: 479 | features: 52


In [26]:
# =============================================================================
# 10B. RANDOM-FOREST FEATURE IMPORTANCE
# =============================================================================

if MODEL_FOR_MAPPING == "random_forest":

    importance_rows = []

    for representation in [
        "polaris",
        "z_plus_polaris",
    ]:

        model = MAPPING_MODELS[
            representation
        ]

        features = REPRESENTATION_FEATURE_NAMES[
            representation
        ]

        for feature, importance in zip(
            features,
            model.feature_importances_,
        ):

            domain = (
                "POLARIS"
                if feature.startswith(
                    "POLARIS__"
                )
                else "IA45_z"
            )

            importance_rows.append(
                {
                    "representation": representation,
                    "feature": feature,
                    "domain": domain,
                    "importance": importance,
                }
            )


    importance_df = pd.DataFrame(
        importance_rows
    )

    importance_df.to_csv(
        OUTPUT_DIR
        / "RF_feature_importance.csv",
        index=False,
    )


    print(
        "Top POLARIS-only features:"
    )

    display(
        importance_df.loc[
            importance_df[
                "representation"
            ].eq(
                "polaris"
            )
        ]
        .sort_values(
            "importance",
            ascending=False,
        )
        .head(
            20
        )
    )


    print(
        "\nTop Z + POLARIS features:"
    )

    display(
        importance_df.loc[
            importance_df[
                "representation"
            ].eq(
                "z_plus_polaris"
            )
        ]
        .sort_values(
            "importance",
            ascending=False,
        )
        .head(
            30
        )
    )


Top POLARIS-only features:


,representation,feature,domain,importance
5,polaris,POLARIS__ph_0_30_dw,POLARIS,0.162148
1,polaris,POLARIS__silt_0_30_dw,POLARIS,0.149881
6,polaris,POLARIS__om_log10_0_30_dw,POLARIS,0.147753
2,polaris,POLARIS__clay_0_30_dw,POLARIS,0.146133
0,polaris,POLARIS__sand_0_30_dw,POLARIS,0.143490
4,polaris,POLARIS__theta_s_0_30_dw,POLARIS,0.126268
3,polaris,POLARIS__bd_0_30_dw,POLARIS,0.124327



Top Z + POLARIS features:


,representation,feature,domain,importance
21,z_plus_polaris,z__constant_NBR2,IA45_z,0.034689
22,z_plus_polaris,z__amp1_B2,IA45_z,0.033231
17,z_plus_polaris,z__constant_NDVI,IA45_z,0.030015
36,z_plus_polaris,z__amp1_NBR2,IA45_z,0.028009
37,z_plus_polaris,z__amp2_B2,IA45_z,0.025756
23,z_plus_polaris,z__amp1_B3,IA45_z,0.025121
50,z_plus_polaris,z__amp2_BSI,IA45_z,0.023231
25,z_plus_polaris,z__amp1_B5,IA45_z,0.023099
48,z_plus_polaris,z__amp2_NDMI,IA45_z,0.022987
47,z_plus_polaris,z__amp2_NDVI,IA45_z,0.022660


## 11. Raster helpers

Annual Z and Z+POLARIS maps are written for 2016–2025.

The POLARIS-only model is static, so it is mapped **once** to the Sentinel grid
rather than redundantly writing ten identical annual rasters.


In [27]:
# =============================================================================
# 11. RASTER HELPERS
# =============================================================================

def get_band_names(
    src,
):
    return [
        (
            str(
                desc
            ).strip()
            if desc is not None
            and str(
                desc
            ).strip()
            else f"band_{i}"
        )
        for i, desc in enumerate(
            src.descriptions,
            start=1,
        )
    ]


def ia_band_indexes(
    src,
):

    band_names = get_band_names(
        src
    )

    lookup = {
        normalize_feature_name(
            name
        ): i
        for i, name in enumerate(
            band_names,
            start=1,
        )
    }

    indexes = []

    for feature in IA_COLS:

        key = normalize_feature_name(
            feature
        )

        if key not in lookup:
            raise KeyError(
                f"Annual raster is missing IA45 band: {feature}"
            )

        indexes.append(
            lookup[
                key
            ]
        )

    return indexes


def predictive_entropy(
    proba,
):
    eps = 1e-12

    return -np.sum(
        proba
        * np.log(
            np.maximum(
                proba,
                eps,
            )
        ),
        axis=1,
    )


def write_prediction_outputs(
    model,
    feature_block,
    valid,
    class_block,
    uncertainty_block,
):

    if not np.any(
        valid
    ):
        return

    X_block = feature_block[
        valid
    ]

    proba = model.predict_proba(
        X_block
    ).astype(
        np.float32,
        copy=False,
    )

    winner_idx = np.argmax(
        proba,
        axis=1,
    )

    winner_class = np.asarray(
        model.classes_,
        dtype=object,
    )[
        winner_idx
    ]

    winner_code = np.fromiter(
        (
            CLASS_TO_CODE[
                str(
                    cls
                )
            ]
            for cls in winner_class
        ),
        dtype=np.uint8,
        count=len(
            winner_class
        ),
    )

    class_block[
        valid
    ] = winner_code

    sorted_p = np.sort(
        proba,
        axis=1,
    )

    uncertainty_block[
        0
    ][
        valid
    ] = sorted_p[
        :,
        -1
    ]

    uncertainty_block[
        1
    ][
        valid
    ] = (
        sorted_p[
            :,
            -1
        ]
        - sorted_p[
            :,
            -2
        ]
    )

    uncertainty_block[
        2
    ][
        valid
    ] = predictive_entropy(
        proba
    ).astype(
        np.float32
    )


## 12. Map Z-only and Z+POLARIS annually


In [28]:
# =============================================================================
# 12. ANNUAL Z / Z + POLARIS MAPPING
# =============================================================================

def classify_annual_representation(
    year,
    representation,
):

    if representation not in {
        "z",
        "z_plus_polaris",
    }:
        raise ValueError(
            representation
        )

    model = MAPPING_MODELS[
        representation
    ]

    src_path = annual_k2_path(
        year
    )

    class_path = (
        OUTPUT_DIR
        / (
            f"{representation}_"
            f"{MODEL_FOR_MAPPING}_{year}_class.tif"
        )
    )

    uncertainty_path = (
        OUTPUT_DIR
        / (
            f"{representation}_"
            f"{MODEL_FOR_MAPPING}_{year}_uncertainty.tif"
        )
    )

    params = reference_lookup[
        year
    ]

    mu = params[
        "mean"
    ].astype(
        np.float32
    )

    sigma = params[
        "sd"
    ].astype(
        np.float32
    )

    year_start = time.perf_counter()

    with rasterio.open(
        src_path
    ) as src, rasterio.open(
        KEEP_MASK
    ) as mask_src, rasterio.open(
        POLARIS_SOURCE
    ) as polaris_src:

        ia_indexes = ia_band_indexes(
            src
        )

        if grids_match_definition(
            polaris_src,
            src.crs,
            src.transform,
            src.width,
            src.height,
        ):

            polaris_reader = polaris_src
            polaris_vrt = None

        else:

            polaris_vrt = WarpedVRT(
                polaris_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=POLARIS_RESAMPLING,
            )

            polaris_reader = polaris_vrt


        if grids_match_definition(
            mask_src,
            src.crs,
            src.transform,
            src.width,
            src.height,
        ):

            mask_reader = mask_src
            mask_vrt = None

        else:

            mask_vrt = WarpedVRT(
                mask_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=Resampling.nearest,
            )

            mask_reader = mask_vrt


        class_profile = src.profile.copy()

        class_profile.update(
            count=1,
            dtype="uint8",
            nodata=0,
            compress=COMPRESS,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        uncertainty_profile = src.profile.copy()

        uncertainty_profile.update(
            count=3,
            dtype="float32",
            nodata=-9999.0,
            compress=COMPRESS,
            predictor=3,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )


        blocks = list(
            src.block_windows(
                1
            )
        )

        with rasterio.open(
            class_path,
            "w",
            **class_profile,
        ) as class_dst, rasterio.open(
            uncertainty_path,
            "w",
            **uncertainty_profile,
        ) as uncertainty_dst:

            class_dst.set_band_description(
                1,
                "predicted_parent_class",
            )

            uncertainty_dst.set_band_description(
                1,
                "max_probability",
            )

            uncertainty_dst.set_band_description(
                2,
                "probability_margin",
            )

            uncertainty_dst.set_band_description(
                3,
                "prediction_entropy",
            )


            for block_number, (
                _,
                window,
            ) in enumerate(
                blocks,
                start=1,
            ):

                keep = mask_reader.read(
                    1,
                    window=window,
                    masked=True,
                ).filled(
                    0
                )

                ia = src.read(
                    ia_indexes,
                    window=window,
                    masked=True,
                ).filled(
                    np.nan
                ).astype(
                    np.float32,
                    copy=False,
                )

                ia = np.moveaxis(
                    ia,
                    0,
                    -1,
                )

                valid = (
                    np.isfinite(
                        ia
                    ).all(
                        axis=2
                    )
                    & np.isfinite(
                        keep
                    )
                    & (
                        keep > 0
                    )
                )

                z = (
                    ia
                    - mu
                ) / sigma


                if representation == "z":

                    features = z


                else:

                    soil_raw = polaris_reader.read(
                        SELECTED_POLARIS_INDEXES,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ).astype(
                        np.float32,
                        copy=False,
                    )

                    soil_raw = np.moveaxis(
                        soil_raw,
                        0,
                        -1,
                    )

                    soil = aggregate_polaris_0_30(
                        soil_raw
                    ).astype(
                        np.float32,
                        copy=False,
                    )

                    valid &= np.isfinite(
                        soil
                    ).all(
                        axis=2
                    )

                    features = np.concatenate(
                        [
                            z,
                            soil,
                        ],
                        axis=2,
                    )


                class_block = np.zeros(
                    (
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    dtype=np.uint8,
                )

                uncertainty_block = np.full(
                    (
                        3,
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    -9999.0,
                    dtype=np.float32,
                )


                write_prediction_outputs(
                    model=model,
                    feature_block=features,
                    valid=valid,
                    class_block=class_block,
                    uncertainty_block=uncertainty_block,
                )


                class_dst.write(
                    class_block,
                    1,
                    window=window,
                )

                uncertainty_dst.write(
                    uncertainty_block,
                    window=window,
                )


                if (
                    block_number == 1
                    or block_number % 25 == 0
                    or block_number == len(
                        blocks
                    )
                ):

                    elapsed = (
                        time.perf_counter()
                        - year_start
                    )

                    print(
                        f"{representation} {year} | "
                        f"{block_number:,}/{len(blocks):,} | "
                        f"{elapsed / 60:.1f} min"
                    )


        if polaris_vrt is not None:
            polaris_vrt.close()

        if mask_vrt is not None:
            mask_vrt.close()


    return {
        "representation": representation,
        "Year": year,
        "class_path": str(
            class_path
        ),
        "uncertainty_path": str(
            uncertainty_path
        ),
        "runtime_minutes": (
            time.perf_counter()
            - year_start
        ) / 60.0,
    }


## 13. Map POLARIS-only once


In [29]:
# =============================================================================
# 13. STATIC POLARIS-ONLY MAPPING
# =============================================================================

def classify_polaris_only():

    model = MAPPING_MODELS[
        "polaris"
    ]

    reference_path = annual_k2_path(
        YEARS_TO_MAP[0]
    )

    class_path = (
        OUTPUT_DIR
        / (
            f"polaris_{MODEL_FOR_MAPPING}_static_class.tif"
        )
    )

    uncertainty_path = (
        OUTPUT_DIR
        / (
            f"polaris_{MODEL_FOR_MAPPING}_static_uncertainty.tif"
        )
    )

    run_start = time.perf_counter()

    with rasterio.open(
        reference_path
    ) as ref, rasterio.open(
        KEEP_MASK
    ) as mask_src, rasterio.open(
        POLARIS_SOURCE
    ) as polaris_src:

        if grids_match_definition(
            polaris_src,
            ref.crs,
            ref.transform,
            ref.width,
            ref.height,
        ):

            polaris_reader = polaris_src
            polaris_vrt = None

        else:

            polaris_vrt = WarpedVRT(
                polaris_src,
                crs=ref.crs,
                transform=ref.transform,
                width=ref.width,
                height=ref.height,
                resampling=POLARIS_RESAMPLING,
            )

            polaris_reader = polaris_vrt


        if grids_match_definition(
            mask_src,
            ref.crs,
            ref.transform,
            ref.width,
            ref.height,
        ):

            mask_reader = mask_src
            mask_vrt = None

        else:

            mask_vrt = WarpedVRT(
                mask_src,
                crs=ref.crs,
                transform=ref.transform,
                width=ref.width,
                height=ref.height,
                resampling=Resampling.nearest,
            )

            mask_reader = mask_vrt


        class_profile = ref.profile.copy()

        class_profile.update(
            count=1,
            dtype="uint8",
            nodata=0,
            compress=COMPRESS,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        uncertainty_profile = ref.profile.copy()

        uncertainty_profile.update(
            count=3,
            dtype="float32",
            nodata=-9999.0,
            compress=COMPRESS,
            predictor=3,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )


        blocks = list(
            ref.block_windows(
                1
            )
        )


        with rasterio.open(
            class_path,
            "w",
            **class_profile,
        ) as class_dst, rasterio.open(
            uncertainty_path,
            "w",
            **uncertainty_profile,
        ) as uncertainty_dst:

            class_dst.set_band_description(
                1,
                "predicted_parent_class",
            )

            uncertainty_dst.set_band_description(
                1,
                "max_probability",
            )

            uncertainty_dst.set_band_description(
                2,
                "probability_margin",
            )

            uncertainty_dst.set_band_description(
                3,
                "prediction_entropy",
            )

            for block_number, (
                _,
                window,
            ) in enumerate(
                blocks,
                start=1,
            ):

                keep = mask_reader.read(
                    1,
                    window=window,
                    masked=True,
                ).filled(
                    0
                )

                soil_raw = polaris_reader.read(
                    SELECTED_POLARIS_INDEXES,
                    window=window,
                    masked=True,
                ).filled(
                    np.nan
                ).astype(
                    np.float32,
                    copy=False,
                )

                soil_raw = np.moveaxis(
                    soil_raw,
                    0,
                    -1,
                )

                soil = aggregate_polaris_0_30(
                    soil_raw
                ).astype(
                    np.float32,
                    copy=False,
                )

                valid = (
                    np.isfinite(
                        soil
                    ).all(
                        axis=2
                    )
                    & np.isfinite(
                        keep
                    )
                    & (
                        keep > 0
                    )
                )


                class_block = np.zeros(
                    (
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    dtype=np.uint8,
                )

                uncertainty_block = np.full(
                    (
                        3,
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    -9999.0,
                    dtype=np.float32,
                )


                write_prediction_outputs(
                    model=model,
                    feature_block=soil,
                    valid=valid,
                    class_block=class_block,
                    uncertainty_block=uncertainty_block,
                )


                class_dst.write(
                    class_block,
                    1,
                    window=window,
                )

                uncertainty_dst.write(
                    uncertainty_block,
                    window=window,
                )


                if (
                    block_number == 1
                    or block_number % 25 == 0
                    or block_number == len(
                        blocks
                    )
                ):

                    print(
                        f"polaris static | "
                        f"{block_number:,}/{len(blocks):,}"
                    )


        if polaris_vrt is not None:
            polaris_vrt.close()

        if mask_vrt is not None:
            mask_vrt.close()


    return {
        "representation": "polaris",
        "Year": "static",
        "class_path": str(
            class_path
        ),
        "uncertainty_path": str(
            uncertainty_path
        ),
        "runtime_minutes": (
            time.perf_counter()
            - run_start
        ) / 60.0,
    }


## 14. Run mapping


In [ ]:
# =============================================================================
# 14. RUN MAPPING
# =============================================================================

mapping_records = []

if RUN_MAPPING:

    static_result = classify_polaris_only()

    mapping_records.append(
        static_result
    )


    for year in YEARS_TO_MAP:

        for representation in [
            "z",
            "z_plus_polaris",
        ]:

            result = classify_annual_representation(
                year=year,
                representation=representation,
            )

            mapping_records.append(
                result
            )


            pd.DataFrame(
                mapping_records
            ).to_csv(
                OUTPUT_DIR
                / "mapping_progress.csv",
                index=False,
            )


    mapping_df = pd.DataFrame(
        mapping_records
    )

    mapping_df.to_csv(
        OUTPUT_DIR
        / "mapping_outputs.csv",
        index=False,
    )

    display(
        mapping_df
    )

else:

    print(
        "RUN_MAPPING is False."
    )


polaris static | 1/336
polaris static | 25/336
polaris static | 50/336
polaris static | 75/336
polaris static | 100/336
polaris static | 125/336
polaris static | 150/336
polaris static | 175/336
polaris static | 200/336
polaris static | 225/336


## 15. Annual class-frequency comparison and map agreement

This directly tests how much adding POLARIS changes the annual Z-only
classification.

POLARIS-only is static, so the same soil-only map is compared against each
annual optical classification.


In [ ]:
# =============================================================================
# 15. CLASS FREQUENCIES / AGREEMENT
# =============================================================================

def class_raster_path(
    representation,
    year=None,
):

    if representation == "polaris":

        return (
            OUTPUT_DIR
            / f"polaris_{MODEL_FOR_MAPPING}_static_class.tif"
        )

    return (
        OUTPUT_DIR
        / (
            f"{representation}_"
            f"{MODEL_FOR_MAPPING}_{year}_class.tif"
        )
    )


frequency_records = []
agreement_records = []


if RUN_MAPPING:

    with rasterio.open(
        class_raster_path(
            "polaris"
        )
    ) as src:

        polaris_map = src.read(
            1,
            masked=False,
        )


    for year in YEARS_TO_MAP:

        maps = {
            "z": None,
            "z_plus_polaris": None,
            "polaris": polaris_map,
        }


        for representation in [
            "z",
            "z_plus_polaris",
        ]:

            with rasterio.open(
                class_raster_path(
                    representation,
                    year,
                )
            ) as src:

                maps[
                    representation
                ] = src.read(
                    1,
                    masked=False,
                )


        for representation, arr in maps.items():

            valid = np.isin(
                arr,
                VEG_PARENT_CODES,
            )

            codes, counts = np.unique(
                arr[
                    valid
                ],
                return_counts=True,
            )

            total = int(
                valid.sum()
            )

            for code_value, count in zip(
                codes,
                counts,
            ):

                frequency_records.append(
                    {
                        "Year": year,
                        "representation": representation,
                        "class_code": int(
                            code_value
                        ),
                        "n_pixels": int(
                            count
                        ),
                        "percent": (
                            100.0
                            * count
                            / total
                        ),
                    }
                )


        pairs = [
            (
                "z",
                "z_plus_polaris",
            ),
            (
                "z",
                "polaris",
            ),
            (
                "z_plus_polaris",
                "polaris",
            ),
        ]


        for a_name, b_name in pairs:

            a = maps[
                a_name
            ]

            b = maps[
                b_name
            ]

            valid_pair = (
                np.isin(
                    a,
                    VEG_PARENT_CODES,
                )
                & np.isin(
                    b,
                    VEG_PARENT_CODES,
                )
            )

            agreement_records.append(
                {
                    "Year": year,
                    "map_a": a_name,
                    "map_b": b_name,
                    "n_paired_pixels": int(
                        valid_pair.sum()
                    ),
                    "exact_agreement": float(
                        np.mean(
                            a[
                                valid_pair
                            ]
                            == b[
                                valid_pair
                            ]
                        )
                    ),
                }
            )


    frequency_df = pd.DataFrame(
        frequency_records
    )

    agreement_df = pd.DataFrame(
        agreement_records
    )


    frequency_df.to_csv(
        OUTPUT_DIR
        / "representation_class_frequencies.csv",
        index=False,
    )

    agreement_df.to_csv(
        OUTPUT_DIR
        / "representation_map_agreement.csv",
        index=False,
    )


    display(
        agreement_df.round(
            4
        )
    )


## 16. Side-by-side classified-map comparison

The same `tab10` 8-class palette used in the first supervised mapping run is
used for all three representations.


In [ ]:
# =============================================================================
# 16. MAP VISUALIZATION
# =============================================================================

PARENT_LABELS = {
    1: "BRTE annual-grass / shrub-bare mix",
    2: "Non-ARTR shrub",
    3: "EF-bare annual-forb",
    4: "Extreme BRTE / EAG",
    5: "Open perennial bunchgrass",
    6: "BRTE-PBG codominant",
    7: "ARTR shrub-steppe",
    8: "EF-PBG codominant",
}


cmap = plt.get_cmap(
    "tab10",
    8,
)

class_norm = BoundaryNorm(
    np.arange(
        0.5,
        9.5,
        1,
    ),
    cmap.N,
)


legend_handles = [
    Patch(
        facecolor=cmap(
            parent - 1
        ),
        edgecolor="none",
        label=(
            f"P{parent} — "
            f"{PARENT_LABELS[parent]}"
        ),
    )
    for parent in VEG_PARENT_CODES
]


if RUN_MAPPING:

    with rasterio.open(
        class_raster_path(
            "polaris"
        )
    ) as src:

        static_polaris = src.read(
            1,
            masked=False,
        )

        bounds = src.bounds


    for year in YEARS_TO_MAP:

        arrays = {}


        for representation in [
            "z",
            "z_plus_polaris",
        ]:

            with rasterio.open(
                class_raster_path(
                    representation,
                    year,
                )
            ) as src:

                arrays[
                    representation
                ] = src.read(
                    1,
                    masked=False,
                )


        arrays[
            "polaris"
        ] = static_polaris


        fig, axes = plt.subplots(
            1,
            3,
            figsize=(
                22,
                8,
            ),
            sharex=True,
            sharey=True,
        )


        panel_titles = {
            "z": "Annual Z only",
            "z_plus_polaris": "Annual Z + POLARIS",
            "polaris": "POLARIS only",
        }


        for ax, representation in zip(
            axes,
            [
                "z",
                "z_plus_polaris",
                "polaris",
            ],
        ):

            arr = np.ma.masked_where(
                ~np.isin(
                    arrays[
                        representation
                    ],
                    VEG_PARENT_CODES,
                ),
                arrays[
                    representation
                ],
            )

            ax.imshow(
                arr,
                cmap=cmap,
                norm=class_norm,
                extent=[
                    bounds.left,
                    bounds.right,
                    bounds.bottom,
                    bounds.top,
                ],
                interpolation="nearest",
            )

            ax.set_title(
                panel_titles[
                    representation
                ],
                fontweight="bold",
            )

            ax.set_xlabel(
                "Easting (m)"
            )

            ax.spines[
                "top"
            ].set_visible(
                False
            )

            ax.spines[
                "right"
            ].set_visible(
                False
            )


        axes[
            0
        ].set_ylabel(
            "Northing (m)"
        )


        fig.suptitle(
            f"Parent classification comparison — {year}",
            fontweight="bold",
        )


        fig.legend(
            handles=legend_handles,
            loc="center left",
            bbox_to_anchor=(
                0.91,
                0.5,
            ),
            frameon=False,
            fontsize=9,
        )


        fig.tight_layout(
            rect=[
                0,
                0,
                0.90,
                0.95,
            ]
        )


        if SAVE_COMPARISON_FIGURES:

            fig.savefig(
                OUTPUT_DIR
                / (
                    f"comparison_Z_POLARIS_"
                    f"{year}.png"
                ),
                dpi=300,
                bbox_inches="tight",
            )


        plt.show()

        plt.close(
            fig
        )


## Interpretation boundary

This is a controlled feature-domain experiment, not a new state definition by fiat.

The central comparison is now whether a **7-dimensional, depth-integrated edaphic
representation** can match or exceed the improvement previously obtained from the
18-dimensional depth-explicit texture representation.

The key tests remain:

1. **Z only vs Z + POLARIS 0–30, grouped OOF**  
   Does the integrated soil context improve discrimination within the observed multi-year domain?

2. **Z only vs Z + POLARIS 0–30, LOYO**  
   Does the static ecological-potential domain improve transfer to an unseen vegetation year?

3. **POLARIS 0–30 only**  
   How much parent-state structure is recoverable from the seven static soil predictors alone?

4. **Confusion structure**  
   Which parent ambiguities are specifically improved or degraded?

5. **Map geography and uncertainty**  
   Does the combined representation improve spatial coherence, probability margin,
   entropy, and temporal persistence without collapsing toward the static soil-only map?

A favorable result is not merely higher accuracy. The stronger pattern would be:

\[
P_{0-30} \ll Z < Z + P_{0-30},
\]

together with higher probability margins, lower predictive entropy, and a modest increase
in temporal persistence.

The previous 18-feature texture-depth experiment should be retained as a baseline rather
than overwritten.


## 17. Optional comparison against the prior 18-feature texture-depth POLARIS run

This cell reads the pooled grouped-OOF and LOYO summaries from the prior POLARIS output
directory if they exist, then reports direct metric differences for the new 0–30 cm
representation. It does not alter either experiment.


In [ ]:
# =============================================================================
# 17. OPTIONAL BASELINE COMPARISON
# =============================================================================

BASELINE_POLARIS_OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_POLARIS"
)

baseline_pooled_path = (
    BASELINE_POLARIS_OUTPUT_DIR
    / "representation_groupedOOF_pooled_metrics.csv"
)

current_pooled_path = (
    OUTPUT_DIR
    / "representation_groupedOOF_pooled_metrics.csv"
)

baseline_loyo_path = (
    BASELINE_POLARIS_OUTPUT_DIR
    / "representation_leaveYearOut.csv"
)

current_loyo_path = (
    OUTPUT_DIR
    / "representation_leaveYearOut.csv"
)


if (
    baseline_pooled_path.exists()
    and current_pooled_path.exists()
):

    baseline_pooled = pd.read_csv(
        baseline_pooled_path
    ).assign(
        polaris_design="18D_texture_depth"
    )

    current_pooled = pd.read_csv(
        current_pooled_path
    ).assign(
        polaris_design="7D_0_30_integrated"
    )

    pooled_design_comparison = pd.concat(
        [
            baseline_pooled,
            current_pooled,
        ],
        ignore_index=True,
    )

    print(
        "Pooled grouped-OOF comparison:"
    )

    display(
        pooled_design_comparison.sort_values(
            [
                "model",
                "representation",
                "polaris_design",
            ]
        ).round(
            4
        )
    )

else:

    print(
        "Pooled baseline comparison skipped; one or both metric files are missing."
    )


if (
    baseline_loyo_path.exists()
    and current_loyo_path.exists()
):

    baseline_loyo = pd.read_csv(
        baseline_loyo_path
    ).assign(
        polaris_design="18D_texture_depth"
    )

    current_loyo = pd.read_csv(
        current_loyo_path
    ).assign(
        polaris_design="7D_0_30_integrated"
    )

    loyo_design_comparison = pd.concat(
        [
            baseline_loyo,
            current_loyo,
        ],
        ignore_index=True,
    )

    print(
        "\nLOYO comparison:"
    )

    display(
        loyo_design_comparison.sort_values(
            [
                "held_out_year",
                "model",
                "representation",
                "polaris_design",
            ]
        ).round(
            4
        )
    )

else:

    print(
        "\nLOYO baseline comparison skipped; one or both metric files are missing."
    )


## 18. True-NCA evaluation

The mapping models are still trained from the full retained vegetation dataset, including
ecologically relevant plots in the analysis buffer. Evaluation below is restricted after
prediction to the true NCA footprint. Grouped-OOF metrics therefore estimate performance
for held-out NCA observations when training support may include plots outside the NCA.


In [ ]:
# =============================================================================
# 18. TRUE NCA GROUPED-OOF EVALUATION
# =============================================================================

import geopandas as gpd

nca = gpd.read_file(
    TRUE_NCA_SHP
)

with rasterio.open(
    class_raster_path(
        "polaris"
    )
) as ref:
    map_crs = ref.crs

if nca.crs != map_crs:
    nca = nca.to_crs(
        map_crs
    )

nca = nca.loc[
    nca.geometry.notna()
    & ~nca.geometry.is_empty
].copy()

try:
    nca_union = nca.geometry.union_all()
except AttributeError:
    nca_union = nca.geometry.unary_union


veg_points = gpd.GeoDataFrame(
    veg.copy(),
    geometry=gpd.points_from_xy(
        veg[
            "Easting"
        ].astype(
            float
        ),
        veg[
            "Northing"
        ].astype(
            float
        ),
    ),
    crs=map_crs,
)

inside_true_nca = np.array(
    [
        nca_union.covers(
            geom
        )
        for geom in veg_points.geometry
    ],
    dtype=bool,
)

print(
    "Common ML rows:",
    len(
        veg_points
    ),
)

print(
    "Inside true NCA:",
    int(
        inside_true_nca.sum()
    ),
)

print(
    "Outside true NCA / buffer-only:",
    int(
        (~inside_true_nca).sum()
    ),
)


nca_oof_records = []
nca_class_records = []

for representation in REPRESENTATIONS:

    pred_all = np.asarray(
        oof_predictions[
            representation
        ][
            MODEL_FOR_MAPPING
        ]
    ).astype(
        str
    )

    y_true_nca = y[
        inside_true_nca
    ]

    y_pred_nca = pred_all[
        inside_true_nca
    ]

    report = classification_report(
        y_true_nca,
        y_pred_nca,
        labels=VEG_CLASSES,
        output_dict=True,
        zero_division=0,
    )

    nca_oof_records.append(
        {
            "representation": representation,
            "model": MODEL_FOR_MAPPING,
            "n_reference_observations": int(
                inside_true_nca.sum()
            ),
            "accuracy": accuracy_score(
                y_true_nca,
                y_pred_nca,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true_nca,
                y_pred_nca,
            ),
            "macro_precision": report[
                "macro avg"
            ][
                "precision"
            ],
            "weighted_precision": report[
                "weighted avg"
            ][
                "precision"
            ],
            "macro_f1": report[
                "macro avg"
            ][
                "f1-score"
            ],
        }
    )

    for parent in VEG_PARENT_CODES:

        cls = f"VEG::{parent}"

        nca_class_records.append(
            {
                "representation": representation,
                "model": MODEL_FOR_MAPPING,
                "parent_class": f"P{parent}",
                "precision": report[
                    cls
                ][
                    "precision"
                ],
                "recall": report[
                    cls
                ][
                    "recall"
                ],
                "f1": report[
                    cls
                ][
                    "f1-score"
                ],
                "support": int(
                    report[
                        cls
                    ][
                        "support"
                    ]
                ),
            }
        )


true_nca_oof_summary = pd.DataFrame(
    nca_oof_records
)

true_nca_class_metrics = pd.DataFrame(
    nca_class_records
)

true_nca_oof_summary.to_csv(
    OUTPUT_DIR
    / "trueNCA_groupedOOF_precision_summary.csv",
    index=False,
)

true_nca_class_metrics.to_csv(
    OUTPUT_DIR
    / "trueNCA_groupedOOF_precision_by_class.csv",
    index=False,
)

print(
    "\nGrouped-OOF performance — TRUE NCA only"
)

display(
    true_nca_oof_summary.round(
        4
    )
)

print(
    "\nClass-specific performance — TRUE NCA only"
)

display(
    true_nca_class_metrics.round(
        4
    )
)


## 19. True-NCA uncertainty comparison

The uncertainty TIFFs contain `max_probability`, `probability_margin`, and
`prediction_entropy`. This cell compares **Z** and **Z + POLARIS 0–30** inside the true
NCA using the absolute theoretical scales rather than per-year min/max normalization.

The primary ambiguity statistic is probability margin,
\(p_{(1)}-p_{(2)}\). Entropy is normalized by \(\log(8)\).


In [ ]:
# =============================================================================
# 19. TRUE NCA UNCERTAINTY SUMMARY — Z VS Z + POLARIS 0–30
# =============================================================================

from rasterio.mask import mask as rio_mask

nca_shapes = [
    geom.__geo_interface__
    for geom in nca.geometry
]


def uncertainty_raster_path(
    representation,
    year,
):
    return (
        OUTPUT_DIR
        / (
            f"{representation}_"
            f"{MODEL_FOR_MAPPING}_{year}_uncertainty.tif"
        )
    )


uncertainty_records = []

for year in YEARS_TO_MAP:

    for representation in [
        "z",
        "z_plus_polaris",
    ]:

        cls_path = class_raster_path(
            representation,
            year,
        )

        unc_path = uncertainty_raster_path(
            representation,
            year,
        )

        with rasterio.open(
            cls_path
        ) as src:

            cls_clip, cls_transform = rio_mask(
                src,
                nca_shapes,
                crop=True,
                filled=True,
                nodata=0,
                all_touched=False,
            )

        with rasterio.open(
            unc_path
        ) as src:

            descriptions = [
                str(
                    x
                ).strip().lower()
                if x is not None
                else ""
                for x in src.descriptions
            ]

            unc_clip, unc_transform = rio_mask(
                src,
                nca_shapes,
                crop=True,
                filled=True,
                nodata=-9999.0,
                all_touched=False,
            )

        if cls_transform != unc_transform:
            raise ValueError(
                f"Class/uncertainty grid mismatch: {representation} {year}"
            )

        lookup = {
            name: i
            for i, name in enumerate(
                descriptions
            )
        }

        required = {
            "max_probability",
            "probability_margin",
            "prediction_entropy",
        }

        if not required.issubset(
            lookup
        ):
            raise KeyError(
                f"Missing uncertainty bands for {representation} {year}: "
                f"{descriptions}"
            )

        cls = cls_clip[
            0
        ]

        pmax = unc_clip[
            lookup[
                "max_probability"
            ]
        ].astype(
            np.float64
        )

        margin = unc_clip[
            lookup[
                "probability_margin"
            ]
        ].astype(
            np.float64
        )

        entropy = unc_clip[
            lookup[
                "prediction_entropy"
            ]
        ].astype(
            np.float64
        )

        valid = (
            np.isin(
                cls,
                VEG_PARENT_CODES,
            )
            & np.isfinite(
                pmax
            )
            & np.isfinite(
                margin
            )
            & np.isfinite(
                entropy
            )
            & (
                pmax >= 0
            )
            & (
                margin >= 0
            )
            & (
                entropy >= 0
            )
        )

        entropy_norm = (
            entropy
            / np.log(
                len(
                    VEG_PARENT_CODES
                )
            )
        )

        uncertainty_records.append(
            {
                "Year": year,
                "representation": representation,
                "n_valid_pixels": int(
                    valid.sum()
                ),
                "mean_pmax": float(
                    np.mean(
                        pmax[
                            valid
                        ]
                    )
                ),
                "mean_margin": float(
                    np.mean(
                        margin[
                            valid
                        ]
                    )
                ),
                "median_margin": float(
                    np.median(
                        margin[
                            valid
                        ]
                    )
                ),
                "mean_normalized_entropy": float(
                    np.mean(
                        entropy_norm[
                            valid
                        ]
                    )
                ),
            }
        )


true_nca_uncertainty = pd.DataFrame(
    uncertainty_records
)

true_nca_uncertainty.to_csv(
    OUTPUT_DIR
    / "trueNCA_uncertainty_summary.csv",
    index=False,
)

display(
    true_nca_uncertainty.round(
        4
    )
)


uncertainty_wide = true_nca_uncertainty.pivot(
    index="Year",
    columns="representation",
    values=[
        "mean_pmax",
        "mean_margin",
        "mean_normalized_entropy",
    ],
)

for metric in [
    "mean_pmax",
    "mean_margin",
    "mean_normalized_entropy",
]:

    uncertainty_wide[
        (
            f"delta_{metric}",
            "z_plus_minus_z",
        )
    ] = (
        uncertainty_wide[
            (
                metric,
                "z_plus_polaris",
            )
        ]
        - uncertainty_wide[
            (
                metric,
                "z",
            )
        ]
    )

print(
    "\nZ + POLARIS minus Z uncertainty changes:"
)

display(
    uncertainty_wide.round(
        4
    )
)


## 20. True-NCA temporal persistence

This cell compares adjacent-year class persistence for `z` and
`z_plus_polaris`. A useful POLARIS contribution should increase persistence modestly without
forcing the annual classifications toward a static soil-only solution.


In [ ]:
# =============================================================================
# 20. TRUE NCA TEMPORAL PERSISTENCE
# =============================================================================

stability_records = []
persistence_summary = []

for representation in [
    "z",
    "z_plus_polaris",
]:

    yearly_arrays = []

    for year in YEARS_TO_MAP:

        with rasterio.open(
            class_raster_path(
                representation,
                year,
            )
        ) as src:

            arr, transform = rio_mask(
                src,
                nca_shapes,
                crop=True,
                filled=True,
                nodata=0,
                all_touched=False,
            )

        yearly_arrays.append(
            arr[
                0
            ]
        )

    stack = np.stack(
        yearly_arrays,
        axis=0,
    )

    valid = np.isin(
        stack,
        VEG_PARENT_CODES,
    )

    n_changes = np.zeros(
        stack.shape[
            1:
        ],
        dtype=np.uint8,
    )

    n_valid_transitions = np.zeros(
        stack.shape[
            1:
        ],
        dtype=np.uint8,
    )

    for i in range(
        len(
            YEARS_TO_MAP
        )
        - 1
    ):

        valid_pair = (
            valid[
                i
            ]
            & valid[
                i + 1
            ]
        )

        changed = (
            stack[
                i
            ]
            != stack[
                i + 1
            ]
        ) & valid_pair

        n_changes += changed.astype(
            np.uint8
        )

        n_valid_transitions += valid_pair.astype(
            np.uint8
        )

        stability_records.append(
            {
                "representation": representation,
                "year_t": YEARS_TO_MAP[
                    i
                ],
                "year_t1": YEARS_TO_MAP[
                    i + 1
                ],
                "n_paired_pixels": int(
                    valid_pair.sum()
                ),
                "exact_agreement": float(
                    np.mean(
                        stack[
                            i
                        ][
                            valid_pair
                        ]
                        == stack[
                            i + 1
                        ][
                            valid_pair
                        ]
                    )
                ),
            }
        )

    persistence = np.full(
        stack.shape[
            1:
        ],
        np.nan,
        dtype=np.float64,
    )

    good = (
        n_valid_transitions
        > 0
    )

    persistence[
        good
    ] = (
        1.0
        - (
            n_changes[
                good
            ]
            / n_valid_transitions[
                good
            ]
        )
    )

    persistence_summary.append(
        {
            "representation": representation,
            "mean_persistence": float(
                np.nanmean(
                    persistence
                )
            ),
            "median_persistence": float(
                np.nanmedian(
                    persistence
                )
            ),
            "mean_number_changes": float(
                np.nanmean(
                    np.where(
                        good,
                        n_changes,
                        np.nan,
                    )
                )
            ),
        }
    )


stability_df = pd.DataFrame(
    stability_records
)

persistence_summary_df = pd.DataFrame(
    persistence_summary
)

stability_df.to_csv(
    OUTPUT_DIR
    / "trueNCA_year_to_year_agreement.csv",
    index=False,
)

persistence_summary_df.to_csv(
    OUTPUT_DIR
    / "trueNCA_persistence_summary.csv",
    index=False,
)

print(
    "Persistence summary:"
)

display(
    persistence_summary_df.round(
        4
    )
)

print(
    "\nAdjacent-year agreement:"
)

display(
    stability_df.round(
        4
    )
)
